In [ ]:
import os
#AA#> Colab Env Variables
os.environ['MPLBACKEND'] = 'notebook'
HF_api_key = os.environ.get("HF_TOKEN")
os.environ['HF_TOKEN'] = HF_api_key

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Extra

In [ ]:
import pandas as pd
eval = pd.read_excel('/path/to/local/data')

In [ ]:
imp = []
for i in range(len(eval)-1):
    row,next = eval.iloc[i],eval.iloc[i+1]
    if row['file'].split('/')[0] in next['file'] and round(row['compFuzzy'],2) <= round(next['compFuzzy'],2):
        imp.append(row['file'])

len(imp)

In [ ]:
import os,shutil
import pandas as pd

#firstDir = '/path/to/local/data'
#ragDir = '/path/to/local/data'
missedDir = '/path/to/local/data'

folders = os.listdir(missedDir)

for folder in folders:
    TARGET_DIR = missedDir + folder
    if '.' not in folder: 
        subDir = os.listdir(TARGET_DIR)
        for x in subDir:
            os.rename(os.path.join(TARGET_DIR, x),
            os.path.join(TARGET_DIR, '0.' + x.split('.')[-1]))


# GenArtist
 - https://github.com/zhenyuw16/GenArtist
 - GenArtist needs python 3.9 vEnv. READ: [Virt Env in Colab](https://stackoverflow.com/questions/48512569/is-it-possible-to-activate-virtualenv-in-google-colab-bin-sh-1-source-not)

In [ ]:
%%bash
#sudo apt-get install python3.10 python3.10-dev
#sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1

sudo apt-get update -y
sudo apt-get install -y python3.9 python3.9-dev python3.9-venv python3.9-distutils


In [ ]:
%%bash
pip install virtualenv
virtualenv -p python3.9 genArtistEnv

source /content/genArtistEnv/bin/activate; python --version

In [ ]:
import os, urllib.request
import zipfile
mainPath = '/content/drive/MyDrive/RAGArtist'
with zipfile.ZipFile(f'{mainPath}/GenArtist-main.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/genArtistEnv/bin')

In [ ]:
%cd /content/genArtistEnv/bin/GenArtist-main/
!source /content/genArtistEnv/bin/activate; pip install pip==23.3

In [ ]:
#AA> Making necessary changes in the scripts

mainDir = "/content/genArtistEnv/bin/GenArtist-main"
textUpdate = {
              'from kornia import SamplePadding': 'from kornia.constants import SamplePadding',
              'map_location="cpu"':'map_location="cuda"'
              }

for origText in textUpdate:
  newText = textUpdate[origText]
  pyFiles = []

  for root, dirs, files in os.walk(mainDir):
    for file in files:
      if file.endswith('.py'):
        if origText in str(open(os.path.join(root, file),'r').read()):
          pyFiles.append(os.path.join(root, file))

  print(f"{len(pyFiles)} with string {origText}")

  for file in pyFiles:
    orig = open(file,'r').read()
    if  origText in str(orig):
      f = open(file,'w')
      udapte = orig.replace(origText, newText)
      f.write(udapte)
      f.close()
      print(f'{file} is updated with text {newText}')


In [ ]:
! source /content/genArtistEnv/bin/activate; pip install -r requirements.txt

In [ ]:
#AA#> Colab Virtual Env requirements
%%bash
source /content/genArtistEnv/bin/activate;
pip install httpx==0.27.2;
pip install groundingdino-py;
pip install matplotlib;
pip install gradio==3.40.0;
pip install pydantic==1.10.21;
pip install k-diffusion;
pip install taming-transformers-rom1504;
pip install ipython;
pip install pyrallis;

In [ ]:
# Pre-trained Models

# Inputs
mainDir = "/content/genArtistEnv/bin/GenArtist-main"
hf = "https://huggingface.co"
preTrMls = {
             f"{hf}/ShilongLiu/GroundingDINO/resolve/main/groundingdino_swint_ogc.pth?download=true":f"{mainDir}/GroundingDINO/weights/groundingdino_swint_ogc.pth",
             f"{hf}/camenduru/AnyDoor/resolve/main/dinov2_vitg14_pretrain.pth?download=true":f"{mainDir}/AnyDoor/path/dinov2_vitg14_pretrain.pth",
             f"{hf}/spaces/xichenhku/AnyDoor/resolve/main/epoch%3D1-step%3D8687.ckpt?download=true":f"{mainDir}/AnyDoor/path/epoch=1-step=8687.ckpt", #16.8 GB
             f"{hf}/spaces/InpaintAI/Inpaint-Anything/resolve/main/pretrained_models/big-lama/config.yaml?download=true": f"{mainDir}/Inpaint-Anything/pretrained_models/big-lama/config.yaml",
             f"{hf}/osunlp/InstructPix2Pix-MagicBrush/resolve/main/MagicBrush-epoch-000168.ckpt?download=true": f"{mainDir}/InstructPix2Pix/checkpoints/MagicBrush-epoch-000168.ckpt", #14.8 GB
             f"{hf}/spaces/InpaintAI/Inpaint-Anything/resolve/main/pretrained_models/big-lama/models/best.ckpt?download=true": f"{mainDir}/Inpaint-Anything/pretrained_models/big-lama/models/best.ckpt",

          }

# prepare progressbar
import progressbar
pbar = None

def show_progress(block_num, block_size, total_size):
    global pbar
    if pbar is None:
        pbar = progressbar.ProgressBar(maxval=total_size)
        pbar.start()

    downloaded = block_num * block_size
    if downloaded < total_size:
        pbar.update(downloaded)
    else:
        pbar.finish()
        pbar = None

# Downloading Models
for hfModel in list(preTrMls.keys()):
  os.makedirs(preTrMls[hfModel][:-4], exist_ok=True)
  urllib.request.urlretrieve(hfModel, preTrMls[hfModel],show_progress)
  print(f"Downloaded {hfModel} to {preTrMls[hfModel]}")

In [ ]:
%cd /content/genArtistEnv/bin/GenArtist-main/
!source /content/genArtistEnv/bin/activate; python demo_t2i_rag.py

In [ ]:
import subprocess
#! zip  -r /content/results.zip /content/genArtistEnv/bin/GenArtist-main/results
subprocess.run (["cp", "-r", "/content/genArtistEnv/bin/GenArtist-main/results" ,f"{mainPath}/Results/RAGArtist"])

In [ ]:
! rm  -r /content/genArtistEnv/bin/GenArtist-main/results/*

In [ ]:
mainPath

# ImageRAG

In [ ]:
! pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import os, urllib.request
import zipfile
with zipfile.ZipFile('ImageRAG-main.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

In [ ]:
%cd /content/ImageRAG-main/
! conda env create -f environment.yml

In [ ]:
with zipfile.ZipFile('/content/OmniGen-main.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

In [ ]:
RagSource = 'Bam_Castle'
os.makedirs(f'datasets/{RagSource}')

In [ ]:
%%shell
eval "$(conda shell.bash hook)" # copy conda command to shell
conda activate ImageRAG

python imageRAG_OmniGen.py \
--prompt "دو خودروی برقی در محوطه ارگ بم و رژه هواپیماها در آسمان" \
--mode "generation" \
--out_name 'Bam_Castle' \
--dataset 'Bam_Castle' \
--omnigen_path /content/OmniGen-main \
--openai_api_key "$OPENAI_API_KEY"

conda deactivate

In [ ]:
! rm /content/ImageRAG-main/datasets/fromWeb1/*

In [ ]:
%%shell
eval "$(conda shell.bash hook)" # copy conda command to shell
conda activate ImageRAG

python imageRAG_SDXL.py \
--prompt "A cat in cradle." \
--out_name "cat_cradle" \
--dataset "example_dataset" \
--openai_api_key "$OPENAI_API_KEY"

conda deactivate

## WebSearch

In [ ]:
unsplash_access = os.environ.get("UNSPLASH_ACCESS_KEY")
unsplash_secret = "IHGsIMN30mgQZZV0TW-_Ip5GsgeBW8RqXEtA2aSvoG8"

In [ ]:
"دو خودروی برقی در محوطه برج میلاد و رژه هواپیماها در آسمان"

## LLama Translator

In [ ]:
%cd /content/

In [ ]:
%%bash
pip install virtualenv
virtualenv llamaTranslate

In [ ]:
! source /content/llamaTranslate/bin/activate; pip install haystack-ai

In [ ]:
!source /content/llamaTranslate/bin/activate;
from haystack.components.generators import HuggingFaceLocalGenerator
import torch

gpuModels = [f"meta-llama/Llama-3.2-{paramCount}B-Instruct" for paramCount in [1,3]]

def llamaGpuModel(model = gpuModels[-1], nCtx = 1024,maxToken = 256,temp = 0.1):
  """
    Downloads Llama GPU model from HF APIs and warms it up
    Developed by <amin.aria@aretum.com>
      Inputs:
        model: GPU Model to download from HF API
        nCtx: Max number of tokes in both prompt and generated answer
        maxToken: Max number of tokens in the generated answer
        temp: Temprature ratio. Higher means more innovation and hallucination

      Outpus:
        llama_chat_gpu: GPU-optimized local Llama model
  """

  llama_chat_gpu = HuggingFaceLocalGenerator(
                                              model=model,
                                              huggingface_pipeline_kwargs={"device_map":"auto",
                                                                            # Lower weight precision (float16 or int8) >> Less RAM needed for storage of 1-7-70B parameters
                                                                            "torch_dtype":torch.bfloat16,
                                                                            "model_kwargs": {"max_position_embeddings": nCtx, # Equivalent of n_ctx
                                                                                             "repetition_penalty": 1.8 # Higher value in [0-2] >> less likely to repeat.
                                                                                            },
                                                                          },


                                              generation_kwargs={
                                                                  "max_new_tokens": maxToken,
                                                                  "temperature": temp, #AA#> Higher temp >> More innovation and hallucination
                                                                  "top_k": 3,
                                                                  "return_full_text": False,
                                                                  "top_p": 0.3 #AA#> Lower value for top_p (token sample rate in LSTM) >> More deterministic
                                                                }
                                             )


  llama_chat_gpu.warm_up()
  print(f'GPU Chat Generator is created. Model Params: {llama_chat_gpu.generation_kwargs}')
  return llama_chat_gpu

### Translator Deployment

In [ ]:
!source /content/llamaTranslate/bin/activate;
llama_gpu = llamaGpuModel( nCtx = 1024,maxToken = 256,temp = 0.1)

In [ ]:
!source /content/llamaTranslate/bin/activate;

sent ="یک خانواده در حال تفریخ در یک بوستان با آب نما و وسایل بازی کودکان ، در اصفهان، ایران"

ans = llama_gpu.run(f'Translate the following sentence from Persian to English. Sentence:{sent}')
ans['replies'][0].split(':')[1].split('\n')[0]

# Evaluation

In [ ]:
import pandas as pd
resPath = '/path/to/local/data'
res = pd.read_csv(resPath)
res['prompt'] = res['prompt'].apply(lambda x: x[7:])
res['file'] = res['file'].apply(lambda x: x.replace('_',';'))
res.columns

In [ ]:
res['edit#'] = res['file'].apply(lambda x: int(x.split('/')[-1][0]))
res['edit#'].describe()

In [ ]:
#AA#> Determining number of concepts per caption
import os
searchDir = '/path/to/local/data'
imageFolders = [y for x in os.listdir(searchDir) if '.DS' not in x for y in os.listdir(searchDir + x+'/')]
print(f'There are {len(imageFolders)} image concepts in total')

res['concept#'] = res['file'].apply(lambda x: len([y for y in imageFolders if x.split('/')[-2] in y]))
res = res[res['concept#'] <= 5]
res['concept#'].value_counts()

In [ ]:
#AA#> Adding Fuzzy CLIP columns
clipCols = ['Color','Shape','Texture']
fuzzyBins = [-1,0.1,.15,.2,.3,.35]
for col in clipCols:
    res[f'{col}_fuzzy'] = pd.cut(res[col],bins=fuzzyBins,labels=range(1,6)) 

#AA#> Calcuate Scaled Complex metric
feats = ['AttributeBinding','Spatial', 'NonSpatial', 'Color_fuzzy', 'Shape_fuzzy','Texture_fuzzy']
res['Complex_scaled'] = res[feats].apply(lambda x: sum(x)/len(feats), axis = 1)
res['Complex_scaled'].hist()

In [ ]:
res[res['edit#'] == 0][['Complex_scaled','edit#','concept#']].corr()

In [ ]:
res = res.sort_values(['prompt','edit#'])
res['ragOnly'] = res.groupby('prompt',as_index = False)['Complex_scaled'].transform('size').map({1:1,2:0})
res['best'] = res.groupby('prompt',as_index = False)['Complex_scaled'].transform('idxmax')
res['best'] = res.apply(lambda row: res.loc[row['best']]['edit#'],axis = 1)

In [ ]:
resNoDup = res.drop_duplicates(['prompt'],keep = 'last')
print(resNoDup['ragOnly'].sum(), 'ragOnly out of ', resNoDup.shape[0])
print(resNoDup['concept#'].value_counts())
print(resNoDup['Complex_scaled'].quantile([0.25,0.5,0.75,0.95]))
resNoDup['best'].value_counts()

In [ ]:
top500 = resNoDup.sort_values(by = ['Complex_scaled'],ascending=False).iloc[0:500]
print(top500['ragOnly'].sum(), 'ragOnly out of ', top500.shape[0])
print(top500['Complex_scaled'].quantile([0.25,0.5,0.75,0.95]))
print(top500['concept#'].value_counts())
top500['best'].value_counts()